# Importing Dependencies

In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

# movie.dat

In [ ]:
movie_data = pd.read_csv("movies.dat", sep = "::", header = None,names = ["MovieID","Title","Genres"], engine = "python",encoding = "latin-1")

In [ ]:
movie_data.head()

In [ ]:
movie_data.shape

In [ ]:
movie_data.info()

In [ ]:
movie_data.isnull().sum()

# ratings.dat (Ratings_file)

In [ ]:
ratings_data = pd.read_csv("ratings.dat", sep = "::",names = ["UserID","MovieID","Rating","Timestamp"], engine = "python")

In [ ]:
ratings_data.head()

In [ ]:
ratings_data.shape

In [ ]:
ratings_data.info()

In [ ]:
ratings_data.isnull().sum()

# users.dat (Users_data)

In [ ]:
users_data = pd.read_csv("users.dat", sep = "::", engine = "python", names = ["UserID","Gender","Age","Occupation","Zip-code"])

In [ ]:
users_data.head()

In [ ]:
users_data.shape

In [ ]:
users_data.info()

In [ ]:
users_data.isnull().sum()

# Merging the dataset

In [ ]:
# Merge movies_data and ratings_data using common column MovieID
movies_ratings = pd.merge(movie_data,ratings_data, on = "MovieID",how = "inner")
movies_ratings

In [ ]:
# Merge users_data and ratings_data using common column UserID
users_ratings = pd.merge(users_data,ratings_data,on="UserID",how = "inner")
users_ratings

In [ ]:
# Merge users_data and ratings_data using common column UserID
users_movie_ratings = pd.merge(users_data,movies_ratings,on="UserID",how = "inner")
users_movie_ratings

# Exploratory Data Analysis

In [ ]:
users_movie_ratings.head()

In [ ]:
users_movie_ratings.shape

In [ ]:
users_movie_ratings.info()

In [ ]:
users_movie_ratings.isna().sum()

In [ ]:
users_movie_ratings.drop(columns = ["Zip-code","Timestamp"],axis =1,inplace=True)

In [ ]:
users_movie_ratings

# Univariant Analysis

In [ ]:
users_movie_ratings.Gender.value_counts().plot(kind = "bar",color = ["Blue","Red"],alpha = 0.5)
users_movie_ratings.Gender.value_counts()

In [ ]:
users_movie_ratings.Age.value_counts().plot(kind = "bar")
users_movie_ratings.Age.value_counts()

In [ ]:
users_movie_ratings["Occupation"].value_counts().plot(kind = "bar", alpha = 0.8)
users_movie_ratings["Occupation"].value_counts()

In [ ]:
users_movie_ratings["Rating"].value_counts().plot(kind = "barh", color = ["blue","green","red","orange","purple"], alpha =  0.6)
users_movie_ratings["Rating"].value_counts()

# Bivariate Analysis

In [ ]:
users_movie_ratings.groupby("Rating")["Gender"].value_counts().plot(kind = "barh", color = ["blue","green","red","orange","purple"], alpha =  0.6)
users_movie_ratings.groupby("Gender")["Rating"].value_counts()

In [ ]:
sns.countplot(x="Rating",hue = "Gender",data = users_movie_ratings)
pd.crosstab(users_movie_ratings["Rating"],users_movie_ratings["Gender"])

In [ ]:
sns.countplot(x="Age", hue="Gender", data= users_movie_ratings)
users_movie_ratings.groupby("Gender")["Age"].value_counts()

In [ ]:
users_movie_ratings.Genres.unique()

# Encoding

In [ ]:
# Split the 'genre' column into multiple columns based on '|'
genre_dummies = users_movie_ratings['Genres'].str.get_dummies('|')

# Concatenate the genre dummies with the original DataFrame
users_movie_ratings = pd.concat([users_movie_ratings, genre_dummies], axis=1)

# Drop the original 'genre' column if needed
users_movie_ratings.drop(columns=['Genres'], inplace=True)

# Display the updated DataFrame
users_movie_ratings


In [ ]:
users_movie_ratings.Animation.sum()

In [ ]:
users_movie_ratings.Animation.value_counts()

In [ ]:
# Calculate the sum for each column
Genres_sum = users_movie_ratings.iloc[:,7:].sum()

# Create a new DataFrame or Series with column names and their sums
Genres_sum_table = pd.DataFrame({"Genre": Genres_sum.index, "Total_Genre_movies" :Genres_sum.values})

# Display the sums DataFrame
Genres_sum_table.sort_values(by = "Total_Genre_movies", ascending = False, inplace = True)

Genres_sum_table


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(Genres_sum_table['Genre'], Genres_sum_table['Total_Genre_movies'], color='skyblue')
plt.xlabel('Genre')
plt.ylabel('Total_Genre_movies')
plt.title('Most viewed movies')
plt.xticks(rotation=90)  # Rotate x-axis labels for better readability
plt.tight_layout()
plt.show()

# Encoding

In [ ]:
u_m_r_encoded = pd.get_dummies(users_movie_ratings, columns = ["Age","Occupation"])
u_m_r_encoded

In [ ]:
# Encoding the Gender variables, Female -0 , Male-1
u_m_r_encoded.Gender.replace({"F":0,"M":1},inplace=True)
u_m_r_encoded

In [ ]:
u_m_r_encoded.info()

# Separating the features and target

In [ ]:
X = u_m_r_encoded.drop(columns=["Title","Rating"],axis=1) 
Y = u_m_r_encoded["Rating"]

In [ ]:
X.shape,Y.shape

# Train_Test_Split

In [ ]:
X_train,X_test,Y_train,Y_test = train_test_split(X,Y,test_size=0.3,random_state=42)
print(X_train.shape,Y_train.shape,X_test.shape,Y_test.shape)

# Model Training

# Decision Tree

In [ ]:
model_dec = DecisionTreeClassifier(class_weight = "balanced",random_state = 42)

In [ ]:
model_dec.fit(X_train,Y_train)

# Model Evaluation

In [ ]:
# Prediction on train
train_preds_dec = model_dec.predict(X_train)

In [ ]:
# Prediction on test
test_preds_dec = model_dec.predict(X_test)

In [ ]:
# print Classification report of train_data
print(classification_report(Y_train,train_preds_dec))

In [ ]:
# print Classification report of test_data
print(classification_report(Y_test,test_preds_dec))

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
model_ran = RandomForestClassifier()
model_ran.fit(X_train,Y_train)
train_preds_ran = model_ran.predict(X_train)
test_preds_ran = model_ran.predict(X_test)

In [ ]:
# print Classification report of train_data
print(classification_report(Y_train,train_preds_ran))

In [ ]:
# print Classification report of train_data
print(classification_report(Y_train,train_preds_ran))